## Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

## Task 1

In [6]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort ascending for horizontal bar (highest at top)

# Step 2: Build your chart
fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',                          # Rule 3: horizontal — long category names
    title='Western Europe scores nearly twice as high as Sub-Saharan Africa in global happiness',
    labels={'Happiness_Score': 'Average Happiness Score (0–10)', 'Region': ''},
    color_discrete_sequence=['#2E75B6']
)

fig.update_layout(
    xaxis=dict(range=[0, 8], gridcolor='#EEEEEE'),   # Rule 1: zero baseline
    yaxis=dict(gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=10, r=30, t=55, b=40),
    height=450
)

fig.update_traces(marker_line_width=0)
fig.show()

## Task 2

In [7]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',                          # Rule 3: horizontal — long country names
    color='Group',
    color_discrete_map={
        'Top 8':    '#2E75B6',                # blue for happiest
        'Bottom 8': '#C00000'                 # red for least happy
    },
    title='The happiest nations score more than twice as high as the least happy — a gap of over 4 points',
    labels={'Happiness_Score': 'Happiness Score (0–10)', 'Country': '', 'Group': ''},
)

# Rule 1: zero baseline
fig.update_layout(
    xaxis=dict(range=[0, 8.5], gridcolor='#EEEEEE'),
    yaxis=dict(gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(orientation='h', y=1.05, x=0),
    margin=dict(l=10, r=30, t=70, b=40),
    height=600
)

fig.update_traces(marker_line_width=0)

# Stretch goal: vertical reference line for global average
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='#888888',
    line_width=1.5
)
fig.add_annotation(
    x=global_avg + 0.08, y=8,              # position label just to the right of line
    text=f'Global avg: {global_avg:.2f}',
    showarrow=False,
    font=dict(color='#888888', size=11),
    xanchor='left'
)

fig.show()

Global average: 5.81


## Task 3

In [9]:
# Stretch goal — grouped bar chart
# ---------------------------------

regions_of_interest = ['Western Europe', 'Latin America and Caribbean',
                        'East Asia', 'Sub-Saharan Africa', 'South Asia']

sub = df[df['Region'].isin(regions_of_interest)]

region_factors = (sub.groupby('Region')[['GDP', 'Freedom']]
                  .mean()
                  .reset_index()
                  .sort_values('GDP'))                # Rule 2: sort by one key factor

region_long = region_factors.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Factor',
    value_name='Score'
)

fig = px.bar(
    region_long,
    x='Score',
    y='Region',
    color='Factor',
    barmode='group',
    orientation='h',
    title='Western Europe leads on both GDP and Freedom — South Asia and Sub-Saharan Africa trail on both',
    labels={'Score': 'Average Sub-factor Score', 'Region': '', 'Factor': ''},
    color_discrete_map={
        'GDP':     '#2E75B6',
        'Freedom': '#70AD47'
    }
)

fig.update_layout(
    xaxis=dict(range=[0, 11], gridcolor='#EEEEEE'),  # Rule 1: zero baseline
    yaxis=dict(gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(orientation='h', y=1.05, x=0),
    margin=dict(l=10, r=30, t=70, b=40),
    height=420
)

fig.update_traces(marker_line_width=0)
fig.show()
